In practise we will use the US census income dataset: 
https://archive.ics.uci.edu/ml/datasets/Census-Income+%28KDD%29

In [ ]:
import pandas as pd
import numpy as np
import shap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix

Reading in the data

In [ ]:
dtypes = [
    ("age", "float64"),
    ("class of worker", "category"),
    ("detailed industry recode", "category"),
    ("detailed occupation recode", "category"),
    ("education", "category"),
    ("wage per hour", "float64"),
    ("enroll in edu inst last wk", "category"),
    ("marital stat", "category"),
    ("major industry code", "category"),
    ("major occupation code", "category"),
    ("race", "category"),
    ("hispanic origin", "category"),
    ("sex", "category"),
    ("member of a labor union", "category"),
    ("reason for unemployment", "category"),
    ("full or part time employment stat", "category"),
    ("capital gains", "float64"),
    ("capital losses", "float64"),
    ("dividends from stocks", "float64"),
    ("tax filer stat", "category"),
    ("region of previous residence", "category"),
    ("state of previous residence", "category"),
    ("detailed household and family stat", "category"),
    ("detailed household summary in household", "category"),
    ("instance weight_ignore", "float64"),
    ("migration code-change in msa", "category"),
    ("migration code-change in reg", "category"),
    ("migration code-move within reg", "category"),
    ("live in this house 1 year ago", "category"),
    ("migration prev res in sunbelt", "category"),
    ("num persons worked for employer", "float64"),
    ("family members under 18", "category"),
    ("country of birth father", "category"),
    ("country of birth mother", "category"),
    ("country of birth self", "category"),
    ("citizenship", "category"),
    ("own business or self employed", "category"),
    ("fill inc questionnaire for veteran's admin", "category"),
    ("veterans benefits", "category"),
    ("weeks worked in year", "float64"),
    ("year", "category"),
    ("targets", "category"),
]

raw_data = pd.read_csv(
    "..\data\census-income.data", names=[d[0] for d in dtypes], dtype=dict(dtypes)
)

edu_code = {
    "Children": 0,
    "Less than 1st grade": 1,
    "1st 2nd 3rd or 4th grade": 2,
    "5th or 6th grade": 3,
    "7th and 8th grade": 4,
    "9th grade": 5,
    "10th grade": 6,
    "11th grade": 7,
    "12th grade no diploma": 8,
    "High school graduate": 9,
    "Some college but no degree": 10,
    "Associates degree-academic program": 11,
    "Associates degree-occup /vocational": 12,
    "Bachelors degree(BA AB BS)": 13,
    "Masters degree(MA MS MEng MEd MSW MBA)": 14,
    "Prof school degree (MD DDS DVM LLB JD)": 15,
    "Doctorate degree(PhD EdD)": 15,
}
raw_data["education-num"] = np.array(
    [edu_code[v.strip()] for v in raw_data["education"]]
).astype("float64")
dtypes.append(("education-num", "float64"))

targets, target_value = pd.factorize(raw_data["targets"])
raw_data = raw_data.drop(columns=["targets"])

Some data cleanup

In [ ]:
# remove some of the data to make it calculate quicker
data = raw_data.drop(
    columns=[
        "instance weight_ignore",
        "detailed industry recode",
        "detailed occupation recode",
        "major industry code",
        "major occupation code",
        "race",
        "hispanic origin",
        "sex",
        "country of birth father",
        "country of birth mother",
        "country of birth self",
        "state of previous residence",
        "detailed household and family stat",
        "education",
    ]
)
data_for_plot = data.copy()
binary_columns = [
    "member of a labor union",
    "live in this house 1 year ago",
    "own business or self employed",
    "fill inc questionnaire for veteran's admin",
    "veterans benefits",
    "year",
]

binary_data = data[binary_columns].copy()
categorical_data = data.select_dtypes(include=["category"]).drop(
    columns=binary_columns, errors="ignore"
)
numerical_data = data.select_dtypes(include=["float64", "int64"]).drop(
    columns=binary_columns, errors="ignore"
)

binary_data[
    (binary_data == " 2")
    | (binary_data == " ?")
    | (binary_data == " Not in universe")
    | (binary_data == " Not in universe under 1 year old")
] = np.nan
binary_data = binary_data.apply(lambda x: pd.factorize(x)[0])

encoder = OneHotEncoder(handle_unknown="ignore")
encoder.fit(categorical_data)

categorical_source = data_for_plot.select_dtypes(include=["category"]).drop(
    columns=binary_columns, errors="ignore"
)

data = pd.concat(
    [
        pd.DataFrame(
            encoder.transform(categorical_source).toarray(),
            index=categorical_source.index,
            columns=encoder.get_feature_names_out(categorical_source.columns),
        ),
        numerical_data,
        binary_data,
    ],
    axis=1,
)

train_x, val_x, train_y, val_y = train_test_split(
    data, pd.Series(targets), random_state=99
)

index = np.random.permutation(
    np.hstack(
        (
            train_y.index.values[train_y == 1],
            np.random.permutation(train_y.index.values[train_y == 0]),
        )
    )[: 2 * np.sum(train_y == 1)]
)
train_x_new = train_x.loc[index]
train_y_new = train_y.loc[index]

data.head()

Building a model...

In [ ]:
my_model = RandomForestClassifier(
    random_state=99, min_samples_split=500, max_leaf_nodes=50, oob_score=True
).fit(train_x_new, train_y_new)
val_pred = my_model.predict(val_x)

print(classification_report(val_y, val_pred, zero_division=0))
print(confusion_matrix(val_y, val_pred))

#ELI5
Calculating featur importance with eli5:

In [ ]:
import eli5
from eli5.sklearn import PermutationImportance

subset_x = val_x.sample(5000)
subset_y = val_y.loc[subset_x.index]

perm = PermutationImportance(my_model, random_state=1, scoring='recall').fit(subset_x, subset_y)
eli5.show_weights(perm, feature_names = data.columns.to_list(), top=50)

#partial dependence plots
Now we will create a couple of partial dependence plots

In [ ]:
from matplotlib import pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

# 1D PDP for education-num (use column index to avoid duplicate column-name issues)
fig, ax = plt.subplots(figsize=(7, 4))
PartialDependenceDisplay.from_estimator(
    estimator=my_model,
    X=subset_x.astype(np.float64),
    features=[103],  # education-num
    kind="average",
    grid_resolution=17,
    ax=ax
)
ax.set_title("Partial Dependence: education-num")
plt.tight_layout()
plt.show()

# Optional: 2D PDP (contour-like) for age vs education-num
fig, ax = plt.subplots(figsize=(7, 5))
PartialDependenceDisplay.from_estimator(
    estimator=my_model,
    X=subset_x.astype(np.float64),
    features=[(96, 103)],  # age, education-num
    kind="average",
    ax=ax
)
ax.set_title("2D Partial Dependence: age vs education-num")
plt.tight_layout()
plt.show()

In [ ]:
import dice_ml


# Model predictions on validation data
val_pred = pd.Series(my_model.predict(val_x), index=val_x.index, name="pred")
val_proba = pd.Series(my_model.predict_proba(val_x)[:, 1], index=val_x.index, name="proba_class_1")

train_for_dice = train_x_new.copy()
train_for_dice["target"] = train_y_new.values

dice_data = dice_ml.Data(
    dataframe=train_for_dice,
    continuous_features=train_x_new.columns.tolist(),
    outcome_name="target",
)
dice_model = dice_ml.Model(model=my_model, backend="sklearn", model_type="classifier")
dice = dice_ml.Dice(dice_data, dice_model, method="random")

dice_binary_cols = [
    c for c in train_x_new.columns
    if set(pd.Series(train_x_new[c]).dropna().unique()).issubset({0, 1})
    ]

query_ids = val_pred.sample(min(3, len(val_pred)), random_state=99).index
dice_rows = []

for qid in query_ids:
    query = val_x.loc[[qid]]
    # Keep 0/1 features constrained as binary for DiCE

    cf_obj = dice.generate_counterfactuals(
        query,
        total_CFs=3,
        desired_class="opposite",
        permitted_range={c: [0, 1] for c in dice_binary_cols},
        verbose=False,
    )
    cf_df = cf_obj.cf_examples_list[0].final_cfs_df
        
    cf_df[dice_binary_cols] = (cf_df[dice_binary_cols] > 0).astype(int)
    diff_cols = cf_df.columns[pd.concat([cf_df, val_x.loc[qid:qid]]).nunique()>1]

    print("original instance (index {}):".format(qid))
    print("target (val_y) for this instance: {}".format(val_y.loc[qid]))
    display(pd.concat([pd.DataFrame(val_x.loc[qid, diff_cols]).T, cf_df[diff_cols]]))
    print("\n\n")


#SHAP
Now we will take a look at the Shapley values for a single row

In [ ]:
row_number = 54
data_for_prediction = val_x.iloc[row_number]  
my_model.predict_proba(data_for_prediction.values.reshape(1, -1))

In [ ]:
explainer = shap.TreeExplainer(my_model)
shap_values = explainer.shap_values(data_for_prediction)
shap_values

SHAP also has nice visualisations for these values

In [ ]:
shap.initjs()
if isinstance(shap_values, list):
    shap_plot = shap.force_plot(explainer.expected_value[1], shap_values[1], data_for_prediction)
else:
    shap_plot = shap.force_plot(explainer.expected_value[1], shap_values[:, 1], data_for_prediction)
display(shap_plot)

For non tree based models we can use the KernelExplainer to get an approximate result

In [ ]:
shap.initjs()
k_explainer = shap.KernelExplainer(my_model.predict, shap.kmeans(val_x, 20))
k_shap_values = k_explainer.shap_values(data_for_prediction)
# Use the same feature vector used to compute k_shap_values (16 model features)
shap.force_plot(k_explainer.expected_value, k_shap_values, data_for_prediction)

This can also be done for multiple the rows

In [ ]:
shap.initjs()
subset_x = val_x.sample(1000)
subset_y = val_y.loc[subset_x.index]
 
explainer = shap.TreeExplainer(my_model)
shap_values = explainer.shap_values(subset_x)
if isinstance(shap_values, list):
    force_plot_obj = shap.force_plot(
        explainer.expected_value[1],
        shap_values[1],
        subset_x
    )
else:
    force_plot_obj = shap.force_plot(
        explainer.expected_value[1],
        shap_values[:, :, 1],
        subset_x
    )
display(force_plot_obj)

We can get a summary of the Shapley values as follows


In [ ]:
shap_values = explainer.shap_values(subset_x)
subset_x_short = subset_x.copy()
subset_x_short.columns = [c[-30:] for c in subset_x_short.columns]

if isinstance(shap_values, list):
	shap.summary_plot(shap_values[1], subset_x_short)
else:
	shap.summary_plot(shap_values[:, :, 1], subset_x_short)

Or for a single feature

In [ ]:
if isinstance(shap_values, list):
    plot_obj = shap.summary_plot(shap_values[0], subset_x, plot_type='bar')
else:
    plot_obj = shap.summary_plot(shap_values[:, :, 0], subset_x, plot_type='bar')
display(plot_obj)

Or for the dependence between 2 features

In [ ]:
if isinstance(shap_values, list):
	plot_obj = shap.dependence_plot(
		'age',
		shap_values[1],
		subset_x,
		interaction_index='education-num',
		x_jitter=1,
		dot_size=20
	)
else:
	plot_obj = shap.dependence_plot(
		'age',
		shap_values[:, :, 1],
		subset_x,
		interaction_index='education-num',
		x_jitter=1,
		dot_size=20
	)
display(plot_obj)